# Used-Car Price Analysis

Use sold-car data to explore price relationships, compare regression models, predict a car's price, and evaluate prediction errors. Make sure `cars.csv` is available in the notebook's working directory.

## 1. Load and Explore the Car Data

Plot each input feature against the sold price and check its correlation with price. Use the plots to judge which relationships look approximately linear.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cars = pd.read_csv("cars.csv")
print("Data shape:", cars.shape)
display(cars.head())

features = ["car_age", "km_thousands", "engine_cc"]
target = "price_lakh"

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature in zip(axes, features):
    ax.scatter(cars[feature], cars[target], alpha=0.6)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)
    ax.set_title(f"{feature} vs. {target}")

plt.tight_layout()
plt.show()

print("Correlation with price_lakh:")
display(cars[features + [target]].corr()[target].drop(target).sort_values(ascending=False).to_frame("correlation"))

Inspect the scatter plots for trends that look roughly straight. Correlation describes the direction and strength of a linear relationship, but does not prove that one feature causes a price change.

## 2. Prepare Features and Split the Data

Use the three car characteristics as `X`, and the sold price as `y`. Keep 20% of the data aside for testing.

In [ ]:
from sklearn.model_selection import train_test_split

X = cars[features]
y = cars[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training cars:", len(X_train))
print("Test cars:", len(X_test))

## 3. Compare Regression Models

Fit each model on the same training data and compare its mean absolute error (MAE) on the test cars. A smaller MAE means predictions were closer to the actual prices on average.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=5),
}

mae_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mae_scores[name] = mean_absolute_error(y_test, predictions)
    print(f"{name}: test MAE = {mae_scores[name]:.2f} lakh")

best_model_name = min(mae_scores, key=mae_scores.get)
print(f"\nSmallest test MAE: {best_model_name} ({mae_scores[best_model_name]:.2f} lakh)")

## 4. Train Linear Regression and Interpret Coefficients

Fit linear regression using all three features. The intercept is the estimated price when all features are zero. Each coefficient estimates the price change, in lakh rupees, for a one-unit increase in that feature while the other features are held fixed. The zero-feature baseline may not describe a realistic car.

In [ ]:
line_model = LinearRegression()
line_model.fit(X_train, y_train)

print(f"Intercept: {line_model.intercept_:.3f} lakh")
print("Coefficients:")
for feature, coefficient in zip(features, line_model.coef_):
    print(f"  {feature}: {coefficient:.4f} lakh per one-unit increase")

print(
    "\nInterpretation: the intercept is the estimated baseline price at zero for all "
    "three features. Each coefficient is the estimated price change for a one-unit "
    "increase in its feature, holding the other two features fixed."
)

## 5. Predict the Price of a New Car

Estimate the price of a 5-year-old car with 60 thousand km driven and a 1300 cc engine.

In [ ]:
new_car = pd.DataFrame({
    "car_age": [5],
    "km_thousands": [60],
    "engine_cc": [1300],
})

predicted_price = line_model.predict(new_car)[0]
print(f"Predicted price: {predicted_price:.2f} lakh rupees")

## 6. Inspect Test-Set Residuals

A residual is the real price minus the predicted price. Positive residuals mean the model predicted too low; negative residuals mean it predicted too high.

In [ ]:
test_predictions = line_model.predict(X_test)

residuals = pd.DataFrame({
    "real_price_lakh": y_test,
    "predicted_price_lakh": test_predictions,
    "residual_lakh": y_test.to_numpy() - test_predictions,
}, index=y_test.index)

display(residuals.head(8).round(2))

## 7. Evaluate Model Errors and Summarize for the Showroom

Calculate test-set MAE, MSE, RMSE, and R². MAE is the average absolute error in lakh rupees; RMSE is also in lakh rupees and gives larger errors more weight. R² describes how much of the test-set price variation the model explains.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mae = mean_absolute_error(y_test, test_predictions)
mse = mean_squared_error(y_test, test_predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, test_predictions)

print(f"MAE:  {mae:.2f} lakh")
print(f"MSE:  {mse:.2f} lakh squared")
print(f"RMSE: {rmse:.2f} lakh")
print(f"R²:   {r2:.3f}")

print("\nShowroom summary:")
print(f"Suggested price for the example car: about {predicted_price:.1f} lakh rupees.")
print(f"The model's average test-set error is about {mae:.1f} lakh rupees (MAE).")